# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [9]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep = "\t")

df_ames.shape

(2930, 82)

In [11]:
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 *df_ames["Half Bath"]

features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms",
            "House Style", "Neighborhood", "Year Built", "SalePrice"]

distance_features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "Year Built", "SalePrice"]
df_ames.loc[[0], distance_features]

,Gr Liv Area,Bedroom AbvGr,Bathrooms,Year Built,SalePrice
0,1656,3,1.0,1960,215000


In [22]:
df_cheaper = df_ames[df_ames["SalePrice"] < df_ames.loc[0, "SalePrice"]].copy()
df_cheaper["distance"] = (((df_cheaper[distance_features] - df_ames.loc[0, distance_features]) ** 2).sum(axis=1)) ** 0.5
df_cheaper.sort_values("distance")[features + ["distance"]]


,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,distance
319,1894,4,2.5,2Story,261.555443
747,2134,5,2.0,2.5Unf,693.126972
2775,1588,3,2.0,1Story,1003.068293
2791,1755,3,2.5,2Story,1005.646185
1472,1536,3,2.0,1Story,1008.050098
...,...,...,...,...,...
2880,480,1,0.0,1Story,179692.848558
2843,498,1,1.0,1Story,180003.728884
726,720,2,1.0,1Story,180102.436677
1553,733,2,1.0,1Story,201902.109930


In [18]:
cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]

X_scaled = X.copy()
X_scaled[cols] = (X_scaled[cols] - X_scaled[cols].mean()) / X_scaled[cols].std()

df_cheaper["euclidean"] = (((X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]) ** 2).sum(axis=1)) ** 0.5

df_cheaper["manhattan"] = abs(X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]).sum(axis=1)

df_cheaper.sort_values("euclidean").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,euclidean,manhattan
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,0,2,2007,WD,Normal,153000,1.0,9.0,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,0,10,2009,WD,Normal,167000,1.0,12.0,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,600,6,2006,WD,Normal,131000,1.0,16.0,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,0,6,2010,WD,Normal,160000,1.0,31.0,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,0,11,2009,COD,Normal,127500,1.0,33.0,0.065281,0.065281


In [19]:
df_cheaper.sort_values("manhattan").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,euclidean,manhattan
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,0,2,2007,WD,Normal,153000,1.0,9.0,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,0,10,2009,WD,Normal,167000,1.0,12.0,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,600,6,2006,WD,Normal,131000,1.0,16.0,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,0,6,2010,WD,Normal,160000,1.0,31.0,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,0,11,2009,COD,Normal,127500,1.0,33.0,0.065281,0.065281


Based on the results, house 319 is the closest cheaper match to house 0 with a distance of 261.56. It has 1,894 square feet, 4 bedrooms, 2.5 bathrooms, and was built in 2002. Its sale price was $214,900. The other matches have much larger distances, so house 319 appears to be the strongest similar cheaper option.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [17]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style"]

X = pd.get_dummies(df_ames[features], columns=["House Style"], dtype=int)

X.head()


df_cheaper = df_ames[df_ames["SalePrice"] < df_ames.loc[0, "SalePrice"]].copy()

df_cheaper["distance"] = (((X.loc[df_cheaper.index] - X.loc[0]) ** 2).sum(axis=1)) ** 0.5

df_cheaper.sort_values("distance").head()[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms",
                                          "House Style", "Neighborhood", "Year Built",
                                          "SalePrice", "distance"]]



,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,Year Built,SalePrice,distance
1927,1657,3,2.0,1Story,NAmes,1970,163500,1.414214
1197,1656,4,2.0,1Story,NWAmes,1973,135000,1.414214
1550,1656,3,1.5,SLvl,IDOTRR,1967,126000,1.500000
2638,1657,4,1.0,1.5Fin,OldTown,1920,111500,2.000000
1293,1656,2,2.0,1.5Fin,OldTown,1940,119164,2.000000


In [23]:
df_cheaper["manhattan"] = abs(X.loc[df_cheaper.index] - X.loc[0]).sum(axis=1)

cols = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X_scaled = X.copy()
X_scaled[cols] = (X_scaled[cols] - X_scaled[cols].mean()) / X_scaled[cols].std()

df_cheaper["euclidean_scaled"] = (((X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]) ** 2).sum(axis=1)) ** 0.5
df_cheaper["manhattan_scaled"] = abs(X_scaled.loc[df_cheaper.index] - X_scaled.loc[0]).sum(axis=1)

In [24]:
df_cheaper.sort_values("manhattan").head()
df_cheaper.sort_values("euclidean_scaled").head()
df_cheaper.sort_values("manhattan_scaled").head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice,Bathrooms,distance,manhattan,euclidean_scaled,manhattan_scaled
1940,1941,535353050,20,RL,75.0,9532,Pave,NaN,Reg,Lvl,...,2,2007,WD,Normal,153000,1.0,62000.001048,9.0,0.017804,0.017804
618,619,534476150,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,...,10,2009,WD,Normal,167000,1.0,48000.002010,12.0,0.023738,0.023738
2700,2701,904100170,20,RL,100.0,21370,Pave,NaN,Reg,Lvl,...,6,2006,WD,Normal,131000,1.0,84000.002119,16.0,0.031651,0.031651
314,315,916125360,20,RL,NaN,57200,Pave,NaN,IR1,Bnk,...,6,2010,WD,Normal,160000,1.0,55000.010045,31.0,0.061324,0.061324
788,789,905450020,20,RL,73.0,9855,Pave,NaN,Reg,Lvl,...,11,2009,COD,Normal,127500,1.0,87500.006314,33.0,0.065281,0.065281


**YOUR RESPONSE HERE**

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

**YOUR RESPONSE HERE**

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [ ]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

We'll want to single out Cal Poly, which we can do like this.

In [ ]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

**YOUR RESPONSE HERE**

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

**YOUR RESPONSE HERE**

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

**YOUR RESPONSE HERE**